In [ ]:
# ============================================================
# BDT CLASSIFICATION WITH XGBOOST — Feature Importance
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, classification_report
import xgboost as xgb                       # XGBoost library



In [ ]:
# ============================================================
# 1. LOAD DATA
# ============================================================

sig_df = pd.read_csv('test_data_signal.csv', header=None)       # Signal
bkg_df = pd.read_csv('test_data_background.csv', header=None)   # Background

# ============================================================
# 2. DATA STRUCTURE
# ============================================================
# Column 0:     Number of objects (particles)
# Columns 1-7:  Event-level features
# Columns 8+:   Groups of 4 per object → (pT, eta, phi, flag)
# Trailing zeros: padding

MAX_OBJECTS = 50               # Max objects to consider
N_FEATURES_PER_OBJ = 4        # pT, eta, phi, flag
EVENT_FEAT_START = 1           # Event features start column
EVENT_FEAT_END = 8             # Event features end column (exclusive)
OBJ_START_COL = 8             # Per-object features start here



In [ ]:
# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================
# For BDTs, we can feed raw padded data OR engineer meaningful features.
# Engineered features are more interpretable for feature importance.

def engineer_features(df, max_objects=MAX_OBJECTS):
    """
    Extract physics-motivated features from raw event data.
    
    Returns:
    --------
    features : np.ndarray of shape (n_events, n_features)
    feature_names : list of str
    """
    raw = df.values                                    # Raw numpy array
    n_events = len(df)
    
    feature_list = []
    feature_names = []
    
    # --- Event-level features (columns 1-7) ---
    event_feats = raw[:, EVENT_FEAT_START:EVENT_FEAT_END]   # (N, 7)
    event_names = [
        'n_objects',           # Column 0 (we'll add separately)
        'event_feat_1',        # Column 1
        'event_feat_2',        # Column 2
        'event_feat_3',        # Column 3
        'event_feat_4',        # Column 4
        'event_feat_5',        # Column 5
        'event_feat_6',        # Column 6
        'event_feat_7'         # Column 7
    ]
    
    # Number of objects
    n_obj_array = raw[:, 0].reshape(-1, 1)             # (N, 1)
    feature_list.append(n_obj_array)
    feature_names.append('n_objects')
    
    # Event-level features directly
    feature_list.append(event_feats)
    feature_names.extend([f'evt_feat_{i}' for i in range(1, 8)])
    
    # --- Per-object features: extract pT, eta, phi, flag arrays ---
    all_pt = np.zeros((n_events, max_objects))
    all_eta = np.zeros((n_events, max_objects))
    all_phi = np.zeros((n_events, max_objects))
    all_flag = np.zeros((n_events, max_objects))
    
    for i in range(n_events):
        n_obj = int(raw[i, 0])
        n_obj = min(n_obj, max_objects)
        for j in range(n_obj):
            col_start = OBJ_START_COL + j * N_FEATURES_PER_OBJ
            if col_start + 3 < raw.shape[1]:
                all_pt[i, j] = raw[i, col_start]
                all_eta[i, j] = raw[i, col_start + 1]
                all_phi[i, j] = raw[i, col_start + 2]
                all_flag[i, j] = raw[i, col_start + 3]
    
    # --- Engineered summary features from per-object data ---
    
    # pT features
    feature_list.append(all_pt[:, 0:1])                # Leading object pT
    feature_names.append('lead_pT')
    feature_list.append(all_pt[:, 1:2])                # Sub-leading object pT
    feature_names.append('sublead_pT')
    feature_list.append(np.sum(all_pt, axis=1, keepdims=True))    # Scalar sum of pT
    feature_names.append('HT')
    feature_list.append(np.mean(all_pt, axis=1, keepdims=True))   # Mean pT
    feature_names.append('mean_pT')
    feature_list.append(np.std(all_pt, axis=1, keepdims=True))    # Std of pT
    feature_names.append('std_pT')
    
    # Eta features
    feature_list.append(all_eta[:, 0:1])               # Leading eta
    feature_names.append('lead_eta')
    feature_list.append(np.mean(all_eta, axis=1, keepdims=True))
    feature_names.append('mean_eta')
    feature_list.append(np.std(all_eta, axis=1, keepdims=True))
    feature_names.append('std_eta')
    
    # Phi features
    feature_list.append(all_phi[:, 0:1])               # Leading phi
    feature_names.append('lead_phi')
    feature_list.append(np.std(all_phi, axis=1, keepdims=True))
    feature_names.append('std_phi')
    
    # Flag features
    feature_list.append(np.sum(all_flag > 0, axis=1, keepdims=True))  # Count flagged objects
    feature_names.append('n_flagged')
    feature_list.append(np.sum(all_flag == 0.1, axis=1, keepdims=True))
    feature_names.append('n_flag_01')
    feature_list.append(np.sum(all_flag == 0.2, axis=1, keepdims=True))
    feature_names.append('n_flag_02')
    
    # Ratios
    sum_pt = np.sum(all_pt, axis=1, keepdims=True)
    sum_pt[sum_pt == 0] = 1e-9                         # Avoid division by zero
    feature_list.append(all_pt[:, 0:1] / sum_pt)       # Leading pT fraction
    feature_names.append('lead_pT_frac')
    feature_list.append(all_pt[:, 1:2] / sum_pt)       # Sublead pT fraction
    feature_names.append('sublead_pT_frac')
    
    # Also include first N object pTs directly (top 5)
    for k in range(5):
        feature_list.append(all_pt[:, k:k+1])
        feature_names.append(f'obj{k+1}_pT')
        feature_list.append(all_eta[:, k:k+1])
        feature_names.append(f'obj{k+1}_eta')
        feature_list.append(all_phi[:, k:k+1])
        feature_names.append(f'obj{k+1}_phi')
        feature_list.append(all_flag[:, k:k+1])
        feature_names.append(f'obj{k+1}_flag')
    
    # Concatenate all features
    features = np.hstack(feature_list)                 # (N, total_features)
    
    return features, feature_names


# Engineer features for signal and background
X_sig, feature_names = engineer_features(sig_df)
X_bkg, _ = engineer_features(bkg_df)

print(f"Signal features: {X_sig.shape}")
print(f"Background features: {X_bkg.shape}")
print(f"Number of engineered features: {len(feature_names)}")
print(f"Feature names: {feature_names}")



In [ ]:

# ============================================================
# 4. COMBINE AND SPLIT
# ============================================================

X = np.concatenate([X_sig, X_bkg], axis=0)             # All features
y = np.concatenate([
    np.ones(len(X_sig)),                               # Signal → 1
    np.zeros(len(X_bkg))                               # Background → 0
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")
print(f"Signal in train: {y_train.sum():.0f}, Background: {len(y_train)-y_train.sum():.0f}")



In [ ]:
# ============================================================
# 5. TRAIN XGBOOST BDT
# ============================================================
# Note: XGBoost does NOT need feature scaling (tree-based)

print("\nTraining XGBoost BDT...")
print("=" * 50)

xgb_clf = xgb.XGBClassifier(
    n_estimators=300,              # Number of boosted trees
    max_depth=5,                   # Maximum tree depth
    learning_rate=0.1,             # Step size shrinkage (eta)
    subsample=0.8,                 # Fraction of samples per tree
    colsample_bytree=0.8,          # Fraction of features per tree
    min_child_weight=5,            # Minimum sum of instance weight in child
    gamma=0.1,                     # Minimum loss reduction for split
    reg_alpha=0.01,                # L1 regularization
    reg_lambda=1.0,                # L2 regularization
    objective='binary:logistic',   # Binary classification
    eval_metric='auc',             # Evaluation metric
    use_label_encoder=False,       # Suppress warning
    random_state=42,
    n_jobs=-1,                     # Use all CPU cores
    verbosity=0
)

# Fit with early stopping using eval set
xgb_clf.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],  # Monitor both
    verbose=False
)




In [ ]:
# ============================================================
# 6. EVALUATE
# ============================================================

y_pred = xgb_clf.predict(X_test)                       # Hard predictions
y_prob = xgb_clf.predict_proba(X_test)[:, 1]           # Class 1 probability

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"\nXGBoost BDT Results:")
print(f"  Accuracy: {acc:.4f}")
print(f"  AUC-ROC:  {auc:.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Background', 'Signal'])}")


In [ ]:
# ============================================================
# 7. ROC CURVE
# ============================================================

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'XGBoost BDT (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate (Background Efficiency)')
plt.ylabel('True Positive Rate (Signal Efficiency)')
plt.title('ROC Curve — XGBoost BDT')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_xgboost.pdf')
plt.show()

In [ ]:

# ============================================================
# 8. FEATURE IMPORTANCE — BUILT-IN (3 TYPES)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# --- 8a. Importance by "weight" (number of times feature is used in splits) ---
importance_weight = xgb_clf.get_booster().get_score(importance_type='weight')
# Map feature indices to names
imp_weight = np.zeros(len(feature_names))
for key, val in importance_weight.items():
    idx = int(key.replace('f', ''))            # 'f0' → 0
    imp_weight[idx] = val

sorted_idx = np.argsort(imp_weight)[-20:]      # Top 20 features
axes[0].barh(np.array(feature_names)[sorted_idx], imp_weight[sorted_idx], color='steelblue')
axes[0].set_xlabel('Weight (# splits)')
axes[0].set_title('Importance: Weight')

# --- 8b. Importance by "gain" (average improvement in loss per split) ---
importance_gain = xgb_clf.get_booster().get_score(importance_type='gain')
imp_gain = np.zeros(len(feature_names))
for key, val in importance_gain.items():
    idx = int(key.replace('f', ''))
    imp_gain[idx] = val

sorted_idx = np.argsort(imp_gain)[-20:]
axes[1].barh(np.array(feature_names)[sorted_idx], imp_gain[sorted_idx], color='darkorange')
axes[1].set_xlabel('Gain (avg loss reduction)')
axes[1].set_title('Importance: Gain')

# --- 8c. Importance by "cover" (average number of samples affected) ---
importance_cover = xgb_clf.get_booster().get_score(importance_type='cover')
imp_cover = np.zeros(len(feature_names))
for key, val in importance_cover.items():
    idx = int(key.replace('f', ''))
    imp_cover[idx] = val

sorted_idx = np.argsort(imp_cover)[-20:]
axes[2].barh(np.array(feature_names)[sorted_idx], imp_cover[sorted_idx], color='green')
axes[2].set_xlabel('Cover (avg # samples)')
axes[2].set_title('Importance: Cover')

plt.suptitle('XGBoost Feature Importance (3 Metrics)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('feature_importance_xgboost.pdf', bbox_inches='tight')
plt.show()


In [ ]:

# ============================================================
# 9. FEATURE IMPORTANCE — SKLEARN-STYLE (based on gain)
# ============================================================

# XGBClassifier also provides sklearn-compatible feature_importances_
importances_sklearn = xgb_clf.feature_importances_     # Normalized gain

# Top 20 most important features
top_n = 20
sorted_idx = np.argsort(importances_sklearn)[-top_n:]

plt.figure(figsize=(10, 8))
plt.barh(
    np.array(feature_names)[sorted_idx],
    importances_sklearn[sorted_idx],
    color='teal'
)
plt.xlabel('Feature Importance (Normalized Gain)')
plt.title(f'Top {top_n} Most Important Features — XGBoost')
plt.tight_layout()
plt.savefig('feature_importance_top20.pdf')
plt.show()

# Print ranking
print("\nFeature Importance Ranking (top 20):")
print("-" * 45)
for rank, idx in enumerate(np.argsort(importances_sklearn)[::-1][:top_n], 1):
    print(f"  {rank:2d}. {feature_names[idx]:<20s} {importances_sklearn[idx]:.4f}")



In [ ]:
# ============================================================
# 10. SHAP VALUES (Most Detailed Explanation)
# ============================================================
# pip install shap

import shap

print("\nComputing SHAP values...")

# TreeExplainer is fast for tree-based models
explainer = shap.TreeExplainer(xgb_clf)

# Compute SHAP values for test set (or a subset for speed)
shap_values = explainer.shap_values(X_test)           # Shape: (n_test, n_features)

# --- 10a. SHAP Summary Plot (beeswarm) ---
# Shows feature importance AND direction of effect
plt.figure()
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=feature_names,
    max_display=20,                  # Show top 20 features
    show=False
)
plt.title('SHAP Summary Plot — XGBoost BDT')
plt.tight_layout()
plt.savefig('shap_summary.pdf', bbox_inches='tight')
plt.show()

# --- 10b. SHAP Bar Plot (global importance) ---
plt.figure()
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=feature_names,
    plot_type='bar',                 # Bar chart of mean |SHAP|
    max_display=20,
    show=False
)
plt.title('SHAP Feature Importance (mean |SHAP value|)')
plt.tight_layout()
plt.savefig('shap_bar.pdf', bbox_inches='tight')
plt.show()

# --- 10c. SHAP Dependence Plots (top 3 features) ---
# Shows how a single feature affects prediction
top3_idx = np.argsort(np.abs(shap_values).mean(axis=0))[-3:][::-1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, feat_idx in enumerate(top3_idx):
    shap.dependence_plot(
        feat_idx,
        shap_values,
        X_test,
        feature_names=feature_names,
        ax=axes[i],
        show=False
    )
plt.suptitle('SHAP Dependence Plots (Top 3 Features)')
plt.tight_layout()
plt.savefig('shap_dependence.pdf', bbox_inches='tight')
plt.show()

# --- 10d. Single event explanation (waterfall plot) ---
# Explain why one specific event was classified as it was
event_idx = 0                                          # Pick first test event
plt.figure()
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[event_idx],
        base_values=explainer.expected_value,
        data=X_test[event_idx],
        feature_names=feature_names
    ),
    max_display=15,
    show=False
)
plt.title(f'SHAP Waterfall — Event {event_idx} (True: {y_test[event_idx]:.0f})')
plt.tight_layout()
plt.savefig('shap_waterfall_example.pdf', bbox_inches='tight')
plt.show()



In [ ]:
# ============================================================
# 11. FEATURE DISTRIBUTIONS: SIGNAL vs BACKGROUND (Top Features)
# ============================================================

# Plot distributions of top 6 most important features
top6_idx = np.argsort(importances_sklearn)[-6:][::-1]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, feat_idx in enumerate(top6_idx):
    ax = axes[i]
    sig_mask = y == 1                                  # Signal events
    bkg_mask = y == 0                                  # Background events
    
    ax.hist(X[sig_mask, feat_idx], bins=50, alpha=0.5,
            density=True, label='Signal', color='blue')
    ax.hist(X[bkg_mask, feat_idx], bins=50, alpha=0.5,
            density=True, label='Background', color='red')
    ax.set_xlabel(feature_names[feat_idx])
    ax.set_ylabel('Normalized')
    ax.legend()
    ax.set_title(f'Rank #{i+1}: {feature_names[feat_idx]}')

plt.suptitle('Signal vs Background — Top 6 Features', fontsize=14)
plt.tight_layout()
plt.savefig('feature_distributions_top6.pdf', bbox_inches='tight')
plt.show()